In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

print("GPUs:", gpus)

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU memory growth enabled")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled


In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv("qoute_dataset.csv")

In [6]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [7]:
df.shape

(3038, 2)

In [8]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [9]:
quotes = quotes.str.lower()

In [10]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [11]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 8978
tokenizer = Tokenizer(num_words=  vocab_size)
tokenizer.fit_on_texts(quotes)

In [13]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [14]:
sequence = tokenizer.texts_to_sequences(quotes)
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [15]:
for i in range(3): 
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [16]:
X = [] 
y = []
for seq in sequence: 
    for i in range(1,len(seq)): 
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [17]:
len(X)

85270

In [18]:
len(y)

85270

***PADDING***

In [19]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen= max_len,padding = 'pre')


In [21]:
y = np.array(y)

In [22]:
X_padded.shape

(85270, 745)

In [23]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes = vocab_size)

In [24]:
y.shape

(85270,)

In [25]:
y_one_hot.shape

(85270, 8978)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [27]:
embedding_dim = 50
rnn_units = 128

In [28]:
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim = vocab_size,output_dim = embedding_dim,input_length = max_len)
)
rnn_model.add(
    SimpleRNN(units=rnn_units,)
)
rnn_model.add(
    Dense(units= vocab_size,activation='softmax')
)

In [29]:
rnn_model.compile(
    optimizer='adam', loss = 'categorocal_crossentropy',metrics=['accuracy']
)

In [30]:
rnn_model.summary

<bound method Model.summary of <keras.engine.sequential.Sequential object at 0x000001F5DF19DF10>>

In [31]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)
lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(
    Dense(units=vocab_size,activation='softmax')
)


In [36]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [37]:
lstm_model.summary

<bound method Model.summary of <keras.engine.sequential.Sequential object at 0x000001F5DF19D4F0>>

In [34]:
X_padded = X_padded.astype("int32")
y_one_hot = y_one_hot.astype("float32")

print("X_padded shape:", X_padded.shape)
print("X_padded dtype:", X_padded.dtype)
print("X_padded size:", X_padded.nbytes / (1024**2), "MB")

print("y_one_hot shape:", y_one_hot.shape)
print("y_one_hot dtype:", y_one_hot.dtype)
print("y_one_hot size:", y_one_hot.nbytes / (1024**2), "MB")

X_padded shape: (85270, 745)
X_padded dtype: int32
X_padded size: 242.3330307006836 MB
y_one_hot shape: (85270, 8978)
y_one_hot dtype: float32
y_one_hot size: 2920.356979370117 MB


In [35]:
import numpy as np

y = np.argmax(y_one_hot, axis=1).astype("int32")

print("y shape:", y.shape)
print("y dtype:", y.dtype)
print("y size:", y.nbytes / (1024**2), "MB")

y shape: (85270,)
y dtype: int32
y size: 0.32527923583984375 MB


In [40]:
epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded,
    y,
    epochs=epochs, 
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 [==============================] - 90s 149ms/step - loss: 4.3598 - accuracy: 0.1886 - val_loss: 6.7009 - val_accuracy: 0.1199
Epoch 2/100
600/600 [==============================] - 91s 152ms/step - loss: 4.2096 - accuracy: 0.2041 - val_loss: 6.7675 - val_accuracy: 0.1176
Epoch 3/100
600/600 [==============================] - 92s 153ms/step - loss: 4.0761 - accuracy: 0.2195 - val_loss: 6.8298 - val_accuracy: 0.1166
Epoch 4/100
600/600 [==============================] - 92s 153ms/step - loss: 3.9372 - accuracy: 0.2382 - val_loss: 6.9137 - val_accuracy: 0.1146
Epoch 5/100
600/600 [==============================] - 92s 153ms/step - loss: 3.7919 - accuracy: 0.2579 - val_loss: 6.9851 - val_accuracy: 0.1115
Epoch 6/100
600/600 [==============================] - 92s 154ms/step - loss: 3.6460 - accuracy: 0.2796 - val_loss: 7.0512 - val_accuracy: 0.1156
Epoch 7/100
600/600 [==============================] - 92s 154ms/step - loss: 3.5033 - accuracy: 0.3014 - val_loss: 7.1285 -

In [41]:
lstm_model.save("lstm_model.h5")

In [42]:
index_to_word = {}
for word,index in word_index.items():
    index_to_word[index] = word

In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
def prediction(model,tokenizer,text,max_len):
    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]
    seq = pad_sequences([seq],maxlen=max_len,padding='pre')

    pred = model.predict(seq,verbose = 0)
    pred_index = np.argmax(pred)

    return index_to_word[pred_index]

In [56]:
seed_text = "i am a"
next_word = prediction(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

badass


In [79]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
    for _ in range(n_words):
        next_word = prediction(model,tokenizer,seed_text,max_len)
        if next_word == "":
            break
        seed_text+=" "+next_word
    return seed_text

In [80]:
seed = "this is my"
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

this is my life for you who you are not my part of


In [82]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [83]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)